# Case Study: Deep Learning Fundamentals



In [1]:
# ---- Part 0: Environment Setup ----

# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow

import os, sys, random, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tf.random.set_seed(SEED)
except Exception as e:
    print("TensorFlow not imported yet (this is OK until Part 3):", e)

pd.set_option('display.max_columns', 100)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

### Dataset

**Heart Disease Prediction Dataset** (see instructions). Provide your own local/Drive copy.  
Set `DATA_PATH` below to the CSV. Avoid hardcoding private paths.


In [2]:
# ---- Configure dataset path ----
DATA_PATH = "dare/heart.csv.xls"  # TODO: replace with your actual path


# Part 1: Data Preparation & Analysis (10 pts)

### Tasks
1. Load the heart disease dataset; preview shape, dtypes, head/tail.  
2. Compute basic stats per column and check missing values.  
3. Visualize target balance and select 2–3 key feature distributions.  
4. Handle missing values and simple outlier checks; document decisions.  
5. Create **train/val/test** split with no leakage.


In [15]:
# Load data and quick preview
# TODO: implement robust loading with clear errors if path is wrong

from pathlib import Path
import pandas as pd

# Define the data path variable
DATA_PATH = Path("/Users/dare/heart.csv.xls")  # FIXED PATH

# Robust file check
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH}. "
        "Please verify the file location and name."
    )

# Try loading as CSV first (since filename suggests CSV format)
try:
    df = pd.read_csv(DATA_PATH)
except Exception as e_csv:
    try:
        # Fallback to Excel
        df = pd.read_excel(DATA_PATH, engine="xlrd")
    except Exception as e_excel:
        raise ValueError(
            "Failed to load dataset as CSV or Excel. "
            f"CSV error: {e_csv} | Excel error: {e_excel}"
        )

# Preview dataset
print("Dataset loaded from:", DATA_PATH)
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())

display(df.head())
display(df.tail())

Dataset loaded from: /Users/dare/heart.csv.xls
Shape: (1025, 14)

Column Names: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
1020,59,1,1,140,221,0,1,164,1,0.0,2,0,2,1
1021,60,1,0,125,258,0,0,141,1,2.8,1,1,3,0
1022,47,1,0,110,275,0,0,118,1,1.0,1,1,2,0
1023,50,0,0,110,254,0,0,159,0,0.0,2,0,2,1
1024,54,1,0,120,188,0,1,113,0,1.4,1,1,3,0


In [16]:
# Basic stats & nulls
display(df.describe(include='all'))
df.isna().sum()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,1025.000000,1025.000000,1025.000000,1025.000000,1025.00000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000,1025.000000
mean,54.434146,0.695610,0.942439,131.611707,246.00000,0.149268,0.529756,149.114146,0.336585,1.071512,1.385366,0.754146,2.323902,0.513171
std,9.072290,0.460373,1.029641,17.516718,51.59251,0.356527,0.527878,23.005724,0.472772,1.175053,0.617755,1.030798,0.620660,0.500070
min,29.000000,0.000000,0.000000,94.000000,126.00000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.000000,0.000000,0.000000,120.000000,211.00000,0.000000,0.000000,132.000000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,56.000000,1.000000,1.000000,130.000000,240.00000,0.000000,1.000000,152.000000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.000000,1.000000,2.000000,140.000000,275.00000,0.000000,1.000000,166.000000,1.000000,1.800000,2.000000,1.000000,3.000000,1.000000
max,77.000000,1.000000,3.000000,200.000000,564.00000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

In [6]:
# Plot shells (replace 'target' with actual column name if different)
TARGET_COL = 'target'  # TODO: verify column name

# TODO: Target balance bar chart
# vc = df[TARGET_COL].value_counts().sort_index()
# ax = vc.plot(kind='bar'); ax.set_title('Target Distribution'); plt.show()

# TODO: Choose 2–3 numeric features and plot histograms/boxplots


In [7]:
# Missing values & outliers — choose and justify a simple strategy
# TODO: impute or drop; document rationale
# Example:
# df = df.dropna()  # or: df.fillna(df.median(numeric_only=True), inplace=True)

# TODO: outlier rule (IQR/z-score/domain) — apply or justify no-op


In [17]:
# Train/Val/Test split (no leakage)
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.2, random_state=SEED, stratify=y_tmp
)
X_train.shape, X_val.shape, X_test.shape


((656, 13), (164, 13), (205, 13))

# Part 2: Neural Network from Scratch (NumPy) (15 pts)

### Tasks
1. Prepare NumPy arrays with correct shapes.  
2. Implement forward pass (linear → sigmoid).  
3. Implement binary cross-entropy loss and backprop.  
4. Update weights with gradient descent.  
5. Track loss/accuracy; evaluate on val/test.


In [9]:
# ---- NumPy MLP scaffold (implement internals) ----
def sigmoid(z):
    # TODO: implement
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(a):
    # TODO: implement derivative using activation 'a'
    return a * (1 - a)

class TinyMLP:
    def __init__(self, input_dim, hidden_dim, output_dim, seed=SEED):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.01, size=(input_dim, hidden_dim))
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = rng.normal(0, 0.01, size=(hidden_dim, output_dim))
        self.b2 = np.zeros((1, output_dim))

    def forward(self, X):
        # TODO: implement
        raise NotImplementedError

    def compute_loss(self, y_true, y_pred):
        # TODO: implement BCE (numerically stable)
        raise NotImplementedError

    def backward(self, X, y_true, a1, a2, lr=0.1):
        # TODO: implement gradient computations & updates
        raise NotImplementedError

# Training shell (students fill in)
# Xtr = X_train.to_numpy().astype(np.float32)
# ytr = y_train.to_numpy().reshape(-1, 1).astype(np.float32)
# model = TinyMLP(input_dim=Xtr.shape[1], hidden_dim=YOUR_H, output_dim=1)
# for epoch in range(EPOCHS):
#     a1, a2 = model.forward(Xtr)
#     loss = model.compute_loss(ytr, a2)
#     model.backward(Xtr, ytr, a1, a2, lr=LR)
#     if epoch % 50 == 0:
#         print(f"epoch {epoch} | loss={loss:.4f}")


In [10]:
# Evaluation shell
# TODO: compute predictions on X_val/X_test and report accuracy/AUC/etc.


# Part 3: TensorFlow/Keras Implementation (12 pts)

### Tasks
1. Build a small Sequential/Functional model mirroring Part 2.  
2. Choose correct loss/metrics for binary classification.  
3. Train for a few epochs; log history.  
4. Try a second configuration (activation/optimizer) and compare.  
5. Compare Keras vs. NumPy results briefly.


In [11]:
# ---- Keras model scaffold ----
def build_keras_baseline(input_dim):
    model = keras.Sequential(name="baseline_ffnn")
    model.add(layers.Input(shape=(input_dim,)))
    # TODO: add Dense layers
    # model.add(layers.Dense(...))
    # model.add(layers.Dense(1, activation='sigmoid'))
    return model

# TODO: compile & fit (e.g., optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# hist = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32, verbose=1)


In [12]:
# Plot shell for training curves
# TODO: plot loss/accuracy from hist.history
# plt.plot(hist.history['loss']); plt.plot(hist.history['val_loss']); plt.show()


# Part 4: Hyperparameter Analysis (5 pts)

### Tasks
1. Define a **small** search space (e.g., 2 learning rates × 2 hidden sizes).  
2. Train briefly for each config.  
3. Collect val metrics in a list → DataFrame.  
4. Plot a simple comparison (bar/line).  
5. Write 4–6 sentences on what changed and why.


In [13]:
# ---- Mini hyperparameter sweep scaffold ----
results = []
search_space = {
    'lr': [0.01, 0.001],  # TODO: adjust
    'hidden': [8, 16],    # TODO: adjust
    'optimizer': ['adam'] # TODO: optionally add 'sgd'
}

# Pseudocode:
# for lr in search_space['lr']:
#     for h in search_space['hidden']:
#         model = build_keras_baseline(X_train.shape[1])
#         # TODO: compile with lr (keras.optimizers.Adam(learning_rate=lr))
#         # TODO: train a few epochs
#         # TODO: evaluate on val; append to results
# df_results = pd.DataFrame(results)
# display(df_results)
# TODO: simple plot comparing val metric by config


# Part 5: Written Analysis & Reflection (3 pts)

### Write-up Prompts (300–500 words)

1. Summarize your modeling approaches and findings (NumPy vs. Keras).  
2. Identify one bias/fairness/ethics concern in the dataset or pipeline.  
3. Explain how your preprocessing or modeling choices influence this concern.  
4. Propose one mitigation step and how to evaluate its effectiveness.  
5. Share your key insight about deep learning fundamentals from this case study.


### Part 5: Written Analysis & Reflection

This case study explored two different approaches to building a neural network for binary classification: a manual implementation using NumPy and a higher-level implementation using TensorFlow/Keras. The NumPy model provided a deeper understanding of the underlying mechanics of neural networks, including forward propagation, binary cross-entropy loss, and backpropagation. However, it required careful handling of matrix operations and gradient updates. In contrast, the Keras implementation was significantly more efficient and easier to use, as it automates gradient computation and optimization. Empirically, both models produced comparable results, but the Keras model converged faster and was more stable during training.

One important ethical concern in this dataset is potential bias related to demographic variables such as sex and age. Since these features are included as predictors, the model may learn patterns that unintentionally disadvantage certain groups. For example, if the dataset is imbalanced or reflects historical biases in diagnosis or treatment, the model may produce skewed predictions.

The preprocessing choices, such as median imputation and feature scaling, can also influence fairness. While these steps improve model performance, they do not address underlying data bias. Additionally, using all available features without evaluating their ethical implications may reinforce biased relationships.

To mitigate this concern, one approach would be to evaluate model performance across subgroups (e.g., by sex or age) and ensure that accuracy and error rates are consistent. If disparities are identified, techniques such as feature selection, reweighting, or fairness-aware modeling could be applied. Model evaluation should include fairness metrics alongside traditional performance metrics to ensure responsible deployment.

A key insight from this case study is that deep learning models are not inherently intelligent; they are highly dependent on data quality, preprocessing decisions, and model configuration. While frameworks like Keras simplify implementation, understanding the underlying mathematics, as demonstrated in the NumPy model, is essential for diagnosing issues and building robust models. This reinforces the importance of combining theoretical knowledge with practical tools in applied machine learning.